# AgriCL-Net, step 2: leaf masks and severity

Uses the trained lesion model from notebook 01 plus **SAM2 (pretrained, no training)** to find the leaf, then
computes severity = lesion pixels / leaf pixels.

**What this measures.** For every image, the same leaf mask is combined once with the *true* lesion mask and
once with the *predicted* one. The gap between the two severities is the lesion model's effect on severity.
It is reported separately for **seen** diseases (PlantSeg test split) and the three **unseen** diseases.
Similar scores for both groups support the project's claim. (True severity, with a true leaf mask, needs the
PDSNets dataset: a later step.)

**Needs:** a finished run of notebook 01 (`best.pt` on Drive) and a T4 GPU.

## Settings

In [ ]:
SIZE = 'l'            # must match the lesion run from notebook 01
IMG_SIZE = 352        # must match the lesion run from notebook 01
LESION_RUN = f"lesion_sam2unet_{SIZE.replace('+', 'p')}_{IMG_SIZE}"   # folder in MyDrive/Colab Notebooks/agricl-net/runs
N_PER_GROUP = 150     # images per group (seen / unseen); None = all
USE_SAM2_LEAF = True  # False: colour-index (ExG) leaf mask, the offline fallback

## GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## PlantSeg v7 dataset (1.1 GB; on Colab downloaded to Drive once)

In [ ]:
import hashlib
ZIP = f'{CACHE}/plantseg.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O "{ZIP}" "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"

h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        h.update(chunk)
assert h.hexdigest() == '9358a66dff88cdd15c4fe009763c40a3', f'bad download, md5 {h.hexdigest()}: delete {ZIP} and rerun'

DATA = f'{WORK}/data/plantseg'
if not os.path.exists(f'{DATA}/Metadata.csv'):
    !unzip -q "{ZIP}" -d "{WORK}/data"

## SAM2-UNet code and SAM2 checkpoint

In [ ]:
!pip -q install hydra-core==1.3.2
SAM2UNET = f'{WORK}/SAM2-UNet'
if not os.path.exists(SAM2UNET):
    !git clone -q https://github.com/WZH0120/SAM2-UNet.git {SAM2UNET}
    !git -C {SAM2UNET} checkout -q 01598e5e9912ffb23f965ecbebf4d1dfecbaa56e

from agricl.models.sam2unet import CHECKPOINTS, CHECKPOINT_URL
CKPT = f'{CACHE}/{CHECKPOINTS[SIZE]}'
if not os.path.exists(CKPT):
    !wget -q --show-progress -O "{CKPT}" "{CHECKPOINT_URL}{CHECKPOINTS[SIZE]}"
print(CKPT, os.path.getsize(CKPT) // 2**20, 'MB')

## Load the trained lesion model and the SAM2 leaf segmenter

In [ ]:
from agricl.pipeline import load_lesion_model
from agricl.leaf import SAM2LeafSegmenter

BEST = f'{DRIVE}/runs/{LESION_RUN}/best.pt'
assert os.path.exists(BEST), f'{BEST} not found: run notebook 01 (with SMOKE_TEST = False) first'
lesion_model = load_lesion_model(SAM2UNET, SIZE, CKPT, BEST, 'cuda')
leaf_segmenter = SAM2LeafSegmenter(SAM2UNET, SIZE, CKPT, device='cuda') if USE_SAM2_LEAF else None

OUT = f"{DRIVE}/runs/severity_{LESION_RUN}_{'sam2' if USE_SAM2_LEAF else 'exg'}_leaf"
os.makedirs(OUT, exist_ok=True)
print('results ->', OUT)

## Severity with true vs. predicted lesions, seen vs. unseen diseases

In [ ]:
import json, random
from agricl.data.plantseg import load_records, split_seen_unseen
from agricl.eval_severity import severity_agreement

seen, unseen = split_seen_unseen(load_records(DATA))
groups = {'seen': [r for r in seen if r.split == 'test'], 'unseen': unseen}
if N_PER_GROUP:
    rng = random.Random(0)
    groups = {k: rng.sample(v, min(N_PER_GROUP, len(v))) for k, v in groups.items()}

summary = severity_agreement(groups, lesion_model, IMG_SIZE, 'cuda', leaf_segmenter,
                             out_csv=f'{OUT}/severity_per_image.csv')
json.dump(summary, open(f'{OUT}/severity_summary.json', 'w'), indent=2)
for g, s in summary.items():
    print(g, json.dumps({k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()}))

## Examples: leaf (green), lesions (red), boxes and severity

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches, numpy as np
from agricl.data.plantseg import load_image
from agricl.pipeline import diagnose_severity

examples = groups['unseen'][:4] + groups['seen'][:2]
fig, ax = plt.subplots(len(examples), 2, figsize=(10, 4 * len(examples)))
for row, r in zip(ax, examples):
    img = load_image(r)
    d = diagnose_severity(img, lesion_model, IMG_SIZE, 'cuda', leaf_segmenter)
    over = np.array(img).astype(float)
    over[d.leaf > 0] = 0.6 * over[d.leaf > 0] + 0.4 * np.array([0, 200, 0])
    over[d.lesion > 0] = [255, 0, 0]
    row[0].imshow(img); row[0].set_title(r.disease)
    row[1].imshow(over.astype(np.uint8))
    row[1].set_title(f'{d.severity.percent:.1f}% {d.severity.level}, {len(d.boxes)} regions')
    for x0, y0, x1, y1 in d.boxes:
        row[1].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='yellow', lw=1))
    for a in row: a.axis('off')
plt.tight_layout(); plt.show()

## Next
* Compare `seen` and `unseen` in the summary: close values support the claim that severity works on unseen diseases.
* Run again with `USE_SAM2_LEAF = False` to compare the SAM2 leaf with the colour-index fallback.
* True severity accuracy (R², RMSE against real leaf labels) comes next, with the PDSNets dataset.